# Fine-Tuning Mistral-7B-Instruct-v0.3 across the Magnificent 7 (NVDA, AAPL, MSFT, AMZN, GOOGL, META, TSLA)

This notebook fine-tunes **Mistral-7B-Instruct-v0.3** using **4-bit QLoRA** on multi-company SEC 10-K filings across multiple fiscal years with:
1. **Strict Citation Enforcement**: Mandatory `[Ticker, Year, Section]` citation anchors across all companies.
2. **Refusal Protocol**: Explicit refusal whenever disclosures lack evidence, eliminating financial hallucinations.
3. **Loss Masking**: Backpropagating gradients exclusively across assistant response tokens.

> **Hardware Requirement**: Google Colab Free Tier **T4 GPU** (16 GB VRAM) or any NVIDIA GPU (A10G, A100, RTX 3090/4090).

## 1. Verify GPU Acceleration
Ensure runtime is set to **T4 GPU** (Runtime -> Change runtime type -> T4 GPU).

In [ ]:
!nvidia-smi

## 2. Install Required Dependencies

In [ ]:
# 1. Uninstall mismatched vision/audio packages (text LLMs do not use them, fixes CUDA ABI conflicts):
!pip uninstall -y torchvision torchaudio

# 2. Install LLM fine-tuning dependencies:
!pip install -q transformers peft bitsandbytes accelerate datasets trl


## 3. Load Multi-Company, Multi-Year ChatML Dataset
Pulls the prepared dataset covering NVIDIA, Apple, Microsoft, Amazon, Alphabet, Meta, and Tesla across FY2023 & FY2024.

In [ ]:
import urllib.request
import json
from datasets import Dataset

# Download full multi-company dataset directly from GitHub repo
DATA_URL = "https://raw.githubusercontent.com/LoneScholar/document-intelligence-rag/main/data/finetune_train.jsonl"
urllib.request.urlretrieve(DATA_URL, "finetune_train.jsonl")

samples = []
with open("finetune_train.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            samples.append(json.loads(line))

dataset = Dataset.from_list(samples)
print(f"Loaded {len(dataset)} multi-company instruction pairs across all Magnificent 7!")
print("Sample pair:", dataset[0])

## 4. Load Base Model in 4-bit NormalFloat (NF4)

In [ ]:
import sys
# 1. Purge any cached references to vision/audio packages:
for mod in list(sys.modules.keys()):
    if mod.startswith("torchvision") or mod.startswith("torchaudio"):
        del sys.modules[mod]

# 2. Tell transformers import-loader to bypass torchvision and torchaudio completely:
import transformers.utils.import_utils as _import_utils
_import_utils.is_torchvision_available = lambda: False
_import_utils.is_torchaudio_available = lambda: False

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

BASE_MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.3"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME, use_fast=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print("Loading Mistral-7B base model into 4-bit VRAM...")
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

model = prepare_model_for_kbit_training(model)
print("Model loaded in 4-bit NF4 successfully!")


## 5. Configure LoRA Parameters

In [ ]:
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

## 6. Train with SFTTrainer and Save Adapter Weights

In [ ]:
from trl import SFTTrainer
# Format ChatML messages into formatted text strings using Mistral template:
def format_prompts(batch):
    texts = []
    for msgs in batch["messages"]:
        texts.append(tokenizer.apply_chat_template(msgs, tokenize=False))
    return {"text": texts}

formatted_dataset = dataset.map(format_prompts, batched=True)
print("Sample formatted prompt:", formatted_dataset[0]["text"][:200])

try:
    from trl import SFTConfig
    training_args = SFTConfig(
        output_dir="./checkpoints/mistral_7b_financial_lora",
        num_train_epochs=3,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        learning_rate=2e-4,
        lr_scheduler_type="cosine",
        warmup_ratio=0.03,
        logging_steps=5,
        save_strategy="epoch",
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        optim="paged_adamw_8bit",
        report_to="none",
        max_length=2048,
        dataset_text_field="text"
    )
    trainer = SFTTrainer(
        model=model,
        train_dataset=formatted_dataset,
        processing_class=tokenizer,
        args=training_args
    )
except (ImportError, TypeError):
    from transformers import TrainingArguments
    training_args = TrainingArguments(
        output_dir="./checkpoints/mistral_7b_financial_lora",
        num_train_epochs=3,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        learning_rate=2e-4,
        lr_scheduler_type="cosine",
        warmup_ratio=0.03,
        logging_steps=5,
        save_strategy="epoch",
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        optim="paged_adamw_8bit",
        report_to="none"
    )
    trainer = SFTTrainer(
        model=model,
        train_dataset=formatted_dataset,
        dataset_text_field="text",
        max_seq_length=2048,
        tokenizer=tokenizer,
        args=training_args
    )

print("Training QLoRA adapters on Mag 7 financial dataset...")
trainer.train()
trainer.model.save_pretrained("./checkpoints/mistral_7b_financial_lora")
tokenizer.save_pretrained("./checkpoints/mistral_7b_financial_lora")
print("Training complete! Adapters saved to ./checkpoints/mistral_7b_financial_lora")


## 7. Multi-Company Inference Verification

In [ ]:
test_prompt = """<|im_start|>system
You are an elite, SEC-compliant Financial Document Intelligence AI Assistant. Rely strictly on provided excerpts and include mandatory [Ticker, Year, Section] citations. If disclosures are insufficient, refuse.<|im_end|>
<|im_start|>user
EXCERPTS FROM VERIFIED SEC 10-K FILINGS:
Citation Anchor: [AMZN, 2024, ITEM 7. MANAGEMENT\x27S DISCUSSION AND ANALYSIS]
Content:
AWS segment sales increased 19% year-over-year to $107.6 billion in 2024, driven by customer adoption of generative AI services including Bedrock and custom Trainium2 chips.

USER QUERY:
What was AWS revenue in fiscal 2024 and what drove its growth?<|im_end|>
<|im_start|>assistant
"""

inputs = tokenizer(test_prompt, return_tensors="pt").to("cuda")
with torch.no_grad():
    outputs = model.generate(**inputs, max_new_tokens=150, temperature=0.1)

print(tokenizer.decode(outputs[0], skip_special_tokens=False))

## 8. Download Fine-Tuned LoRA Adapter to Local Machine
Packages the fine-tuned adapter weights into a zip archive and prompts an automatic browser download.

In [ ]:
!zip -r mistral_7b_financial_lora.zip checkpoints/mistral_7b_financial_lora
from google.colab import files
files.download("mistral_7b_financial_lora.zip")
print("Downloaded mistral_7b_financial_lora.zip! Unzip into ./checkpoints/ in your local repo.")